
<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apriori3d/ico/blob/main/src/examples/ico_basics.ipynb)

**Introduction to ICO Framework**

</div>

# ICO Framework: Basic Examples


---

## 📚 Table of Contents

1. **[Operators](#1-operators)**
   - Creating operators with decorators
   - Creating operators through classes
   
2. **[Operator Chains](#2-operator-chains)**

3. **[Flow Visualisation](#3-flow-visualization)**

4. **[Pipelines](#4-pipelines)**
   - Sequential processing of the same type
   - Operators Name and Signature inference

5. **[Data Streams](#5-data-streams)**
   - Processing collections through iterators
   - Lazy evaluation
   - Batch processing and reduction examples

6. **[Special cases of Operators: Source and Sink](#6-special-cases-of-operators:-source-and-sink)**

7. **[Context-aware operator](#7-context-aware-operator)**

8. **[Type Checking with MyPy](#8-type-checking-with-mypy)**
   - Static type analysis
   - IDE integration
   - Development-time validation

9. **[What's Next?](#9-whats-next)**

---

### Install dependencies if running in Google Colab

In [1]:
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Running in Google Colab - installing dependencies...")

    # Install Apriori ICO framework from GitHub
    %pip install --upgrade -q git+https://github.com/apriori3d/ico.git

    print("✅ Dependencies installed successfully!")
    print("🎯 Ready to run ICO linear regression example!")
else:
    print("📱 Running locally - assuming dependencies are already installed")

📱 Running locally - assuming dependencies are already installed


---

## 1. Operators

```IcoOperator``` is the fundamental building block of ICO Framework. It accept input data of one type and produce output data of another (or the same) type.

ICO automatically infers types from function signatures, allowing for complete typing without additional annotations.

### Creating Operators

### Method 1: Using decorator

In [2]:
from ico import operator


@operator()
def double(x: int) -> int:
    """Double the input value."""
    return x * 2


@operator()
def to_string(x: int) -> str:
    """Convert integer to string."""
    return str(x)


@operator()
def add_prefix(text: str) -> str:
    """Add a prefix to the text."""
    return f"Result: {text}"


print("✅ Created operators using decorator:")
print(f"{double(5)=}")
print(f"{to_string(42)=}")
print(f"{add_prefix('hello')=}")

✅ Created operators using decorator:
double(5)=10
to_string(42)='42'
add_prefix('hello')='Result: hello'


### Method 2: Using IcoOperator class directly

In [3]:
import math

from ico import IcoOperator

sqrt_op = IcoOperator[float, float](math.sqrt, name="square_root")
abs_op = IcoOperator[float, float](abs, name="absolute")

# Method 3: Using lambda functions
multiply_by_3 = IcoOperator[float, float](lambda x: x * 3, name="multiply_by_3")
is_positive = IcoOperator[float, bool](lambda x: x > 0, name="is_positive")

print("✅ Created operators using IcoOperator class:")
print(f"{sqrt_op(16.0)=}")
print(f"{abs_op(-5)=}")
print(f"{multiply_by_3(7)=}")
print(f"{is_positive(-3)=}")

✅ Created operators using IcoOperator class:
sqrt_op(16.0)=4.0
abs_op(-5)=5
multiply_by_3(7)=21
is_positive(-3)=False


---

## 2. Operator Chains
**Operator chains** allow composing operators using the `|` (pipe) operator. The output of one operator automatically becomes the input of the next.

ICO signature: `I → O → O2 = I → O2`

In [4]:
# Direct pipe usage: int → int → str → str
# Pipeline: double → to_string → add_prefix
print("✅ Simple operator chain:")
flow = double | to_string | add_prefix
print(f"{flow(21)=}")  # (21 * 2) → "42" → "Result: 42"

✅ Simple operator chain:
flow(21)='Result: 42'


---

## 3. Flow Visualization

Any flow in ICO can be **visualized** using the `describe()` method. This provides you with:
- **Processing logic**: See how data flows through each operator
- **Type information**: Understand input and output data types at each step  
- **Computation structure**: Visualize the complete transformation pipeline

This makes debugging and understanding complex flows much easier!

In [5]:
# Show the chain signature and visualization
flow.describe()

──────────────────────────────────────── Flow plan: IcoChain | IcoOperator ────────────────────────────────────────

 Flow        Signature  Name 
 double      int → int       
 to_string   int → str       
 add_prefix  str → str      

---

## 4. Pipelines

```IcoPipeline``` is designed for sequential processing of data of the same type. All operators in the pipeline have the signature `I → I`.

ICO signature: `I → I` (through multiple `I → I` steps)

### Direct pipeline creation and usage (int → int)

In [6]:
from ico import IcoPipeline

test_data = [-5, 10, 30, 60, 80]

pipeline = IcoPipeline(
    lambda x: max(x, 0),
    lambda x: x * 2,
    lambda x: min(x, 100),
)
pipeline.describe()

────────────────────────────────────────────── Flow plan: streamline ──────────────────────────────────────────────

 Flow      Signature  Name 
 <lambda>  Any → Any       
 <lambda>  Any → Any       
 <lambda>  Any → Any      

You can see, that types in pipeline are Any, because we do not specify them explicitly and used lambdas without type annotations.

To make pipeline more type-safe, we can explicitly define operators:


In [19]:
pipeline = IcoPipeline(
    IcoOperator[int, int](lambda x: max(x, 0)),
    IcoOperator[int, int](lambda x: x * 2),
    IcoOperator[int, int](lambda x: min(x, 100)),
)
pipeline.describe()

# The pipeline type will be infered from the operators:
print(f"{pipeline.signature=}")  # (int → int)

────────────────────────────────────────────── Flow plan: streamline ──────────────────────────────────────────────

 Flow      Signature  Name 
 <lambda>  int → int       
 <lambda>  int → int       
 <lambda>  int → int      

pipeline.signature=int → int


To make it more concise, we can use the ```@operator``` decorator with function definitions.

This approach gives us a fully typed pipeline with clear operator definitions and a nice visualization when we call ```.describe()```.

All signatures are inferred correctly from function definitions and even names are preserved in the visualization.


In [8]:

@operator()
def max_fn(x: int) -> int:
    return max(x, 0)


@operator()
def double_fn(x: int) -> int:
    return x * 2


@operator()
def min_fn(x: int) -> int:
    return min(x, 100)


pipeline = IcoPipeline(
    max_fn,
    double_fn,
    min_fn,
)
pipeline.describe()



────────────────────────────────────────────── Flow plan: streamline ──────────────────────────────────────────────

 Flow       Signature  Name 
 max_fn     int → int       
 double_fn  int → int       
 min_fn     int → int      

In [9]:
# Apply pipeline directly - validate → multiply → clamp
for value in test_data:
    result = pipeline(value)
    print(f"{pipeline(value)=}  # max(0,{value}) * 2, clamped to 100")

pipeline(value)=0  # max(0,-5) * 2, clamped to 100
pipeline(value)=20  # max(0,10) * 2, clamped to 100
pipeline(value)=60  # max(0,30) * 2, clamped to 100
pipeline(value)=100  # max(0,60) * 2, clamped to 100
pipeline(value)=100  # max(0,80) * 2, clamped to 100


---

## 5. Data Streams

```IcoStream``` allows applying operators to data collections through iterators. This provides lazy evaluation and efficient processing of large data volumes.

ICO signature: `Iterator[I] → Iterator[O]`

You will see that the signature of the ```scale_by_10``` stream becomes ```Iterator[int] → Iterator[int]```.
So operator scale_by_10 becomes a stream operator that takes an iterator of integers and produces an iterator of integers, where each integer is multiplied by 10.

In [20]:
@operator()
def scale_by_10(x: int) -> int:
    return x * 10


scale_stream = scale_by_10.stream()
scale_stream.describe()


───────────────────────────────────────────────── Flow plan: None ─────────────────────────────────────────────────

 Flow                           Signature                      Name 
 ╭── for each in 🎞️ IcoStream()  Iterator[int] → Iterator[int]       
 │   scale_by_10                int → int                           
 ╰─▸ yield                      Iterator[int]

In [11]:
# Direct stream usage - apply operator to each element
data = [1, 2, 3, 4, 5]
result = list(scale_stream(iter(data)))
print(f"Scaled data: {result}")

Scaled data: [10, 20, 30, 40, 50]


### Lazy evaluation of streams

In [12]:
print("\n🧙 Lazy evaluation demonstration:")

@operator()
def expensive_operation(x: int) -> int:
    print(f"  Processing {x}...")  # This shows when the operation is actually called
    return x


# Create lazy flow and iterator
lazy_iterator = expensive_operation.stream()(iter([1, 2, 3]))

print("Stream created, but no processing yet!")

print("Now consuming first element:")
first_result = next(lazy_iterator)
print("First result:", first_result)

print("Processing remaining elements:")
remaining_results = list(lazy_iterator)
print("Remaining results:", remaining_results)


🧙 Lazy evaluation demonstration:
Stream created, but no processing yet!
Now consuming first element:
  Processing 1...
First result: 1
Processing remaining elements:
  Processing 2...
  Processing 3...
Remaining results: [2, 3]


### Reduction example
Useful for batch processing in data loading.

Here we use ```.stream()``` to wrap ```find_batch_max()``` operator into Iterator interface.

The signature of stream is ```Iterator[list[float]] → Iterator[float]```

It applies find_batch_max to each item: ```list[float] → float``` and returns results as ```Iterator[float]```

In [13]:
@operator()
def find_batch_max(numbers: list[float]) -> float:
    """Find maximum value in a batch - useful for normalization in data loading."""
    return max(numbers) if numbers else 0.0


# Simulate data loading batches
batch_data = [
    [1.5, 2.3, 0.8, 3.1],  # Batch 1
    [4.2, 1.9, 3.7, 2.1],  # Batch 2
    [0.5, 5.1, 2.8, 1.3],  # Batch 3
    [3.9, 2.4, 4.6, 1.7],  # Batch 4
]

# Process batches directly with stream (useful for dynamic normalization)
find_batch_max_flow = find_batch_max.stream()
find_batch_max_flow.describe()



───────────────────────────────────────────────── Flow plan: None ─────────────────────────────────────────────────

 Flow                           Signature                                Name 
 ╭── for each in 🎞️ IcoStream()  Iterator[list[float]] → Iterator[float]       
 │   find_batch_max             list[float] → float                           
 ╰─▸ yield                      Iterator[float]

In [14]:
max_values = find_batch_max_flow(iter(batch_data))

print("📊 Batch processing example (data loading scenario):")
for i, max_val in enumerate(max_values):
    print(f"Batch {i}: {batch_data[i]} → max = {max_val}")

print(f"\nBatch max finder signature: {find_batch_max.signature.format()}")
print("\n💡 Use case: Dynamic normalization in data loading pipelines")
print("   - Each batch gets normalized by its own maximum value")
print("   - Prevents gradient explosion in neural network training")
print("   - Stream processing enables memory-efficient handling of large datasets")

📊 Batch processing example (data loading scenario):
Batch 0: [1.5, 2.3, 0.8, 3.1] → max = 3.1
Batch 1: [4.2, 1.9, 3.7, 2.1] → max = 4.2
Batch 2: [0.5, 5.1, 2.8, 1.3] → max = 5.1
Batch 3: [3.9, 2.4, 4.6, 1.7] → max = 4.6

Batch max finder signature: list[float] → float

💡 Use case: Dynamic normalization in data loading pipelines
   - Each batch gets normalized by its own maximum value
   - Prevents gradient explosion in neural network training
   - Stream processing enables memory-efficient handling of large datasets


---

## 6. Special cases of Operators: Source and Sink
The typical pattern in ML pipelines is to define `IcoSource` and `IcoSink`:
-  `IcoSource` has a signature  `()` → `Iterator[Output]`. It takes nothing (`None`) and produces a stream of input items for the pipeline
- `IcoSink` has a signature `Iterator[int]` → `()`. It consumes a stream of input items and returns nothing (`None`)
- Together with intermediate operators they form a flow with signature `()` → `()`. This is a closure - a self-sufficient computation flow.

In [15]:
from collections.abc import Iterator

from ico import sink, source


@source()
def numbers() -> Iterator[int]:
    yield from range(5)


@operator()
def square(x: int) -> int:
    return x * x


@sink()
def print_result(x: int) -> None:
    print(f"Saving: {x}")


closure = numbers | square.stream() | print_result
closure.describe()

print(f"{closure.signature=}")

────────────────────────────────────────── Flow plan: IcoChain | IcoSink ──────────────────────────────────────────

 Flow                           Signature                      Name 
 📚IcoSource(numbers)           () → Iterator[int]                  
 ╭── for each in 🎞️ IcoStream()  Iterator[int] → Iterator[int]       
 │   square                     int → int                           
 ╰─▸ yield                      Iterator[int]                       
 🏁IcoSink(print_result)        Iterator[int] → ()                 

closure.signature=() → ()


---

## 7. Context-aware operator
All examples above have operators with the signature Input → Output, but what about Context?

In ICO, Context doesn't exist alone; it is applied over some data input in ```IcoContextOperator```. 

The signature of such an operator becomes: Input, Context → Context.

As a real-world example, take a look at [Linear Regression](src/examples/ml/ico_linear_regression.ipynb).

In linear regression, Context is a model, and input is a collection of input data samples. By applying a learning operator, we fit the model to the given samples.

### Toy example for IcoContextOperator


### Sum of numbers
Context here is simply a number

In [16]:
from collections.abc import Iterable

from ico import IcoContextOperator

input = range(10)
sum_op = IcoContextOperator[Iterable[int], int, int](sum)

print(f"{sum_op(input, 0)=}")


sum_op(input, 0)=45


### Running average
Context here is a tuple of total and count

In [17]:
from ico import context_operator


@context_operator()
def running_avg(xs: Iterable[int], state: tuple[int, int]) -> tuple[int, int]:
    total, count = state
    for x in xs:
        total += x
        count += 1
    return total, count

state = (0, 0)
state = running_avg(range(10), state)
total, count = state

print(f"Average: {total / count}")

Average: 4.5


---

## 8. Type Checking with MyPy

**MyPy** provides static type analysis for ICO operators. This allows detecting type errors during development, before code execution.

### IDE Integration

MyPy integrates with:
- **VS Code** (via Python extension)
- **PyCharm**
- **Vim/Neovim** (via LSP)
- **Emacs** and other IDEs

Validation works **in real-time** while writing code!

In [18]:
import os
import subprocess
import tempfile

print("🎯 Simple MyPy Demonstration")
print("=" * 40)

# ✅ Correct example: int → int → str
correct_code = """
from ico import operator

@operator()
def square(x: int) -> int:
    return x * x

@operator()
def to_str(x: int) -> str:
    return str(x)

# ✅ Types match: int → int → str
valid_chain = square | to_str
"""

# ❌ Incorrect example: int → str → int (type mismatch)
incorrect_code = """
from ico import operator

@operator()
def square(x: int) -> int:
    return x * x

@operator()
def to_str(x: int) -> str:
    return str(x)

# ❌ Type mismatch: int → int, then str → int (impossible!)
broken_chain = square | to_str | square
"""


def check_with_mypy(code: str, name: str):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        temp_file = f.name

    try:
        result = subprocess.run(
            ["python", "-m", "mypy", "--no-error-summary", temp_file],
            capture_output=True,
            text=True,
            timeout=10,
        )

        print(f"\n{name}:")
        if result.returncode == 0:
            print("✅ MyPy: No type errors")
        else:
            print("❌ MyPy found type errors:")
            for line in result.stdout.strip().split("\n"):
                if "error:" in line.lower():
                    print(f"   {line.strip()}")

    except Exception as e:
        print(f"⚠️  MyPy unavailable: {e}")
    finally:
        os.unlink(temp_file)


# Check both examples
check_with_mypy(correct_code, "🟢 VALID chain: square | to_str")
check_with_mypy(incorrect_code, "🔴 BROKEN chain: square | to_str | square")

print("\n💡 MyPy prevents runtime type errors in ICO chains!")

🎯 Simple MyPy Demonstration

🟢 VALID chain: square | to_str:
✅ MyPy: No type errors

🔴 BROKEN chain: square | to_str | square:
❌ MyPy found type errors:
   /var/folders/wl/r0pchspd2w5d_pyph8bhc0c80000gn/T/tmp5jzsspl5.py:13: error: Unsupported operand types for | ("IcoOperator[int, str]" and "IcoOperator[int, int]")  [operator]

💡 MyPy prevents runtime type errors in ICO chains!


### 🛠️ MyPy Integration Setup

**Install MyPy:**
```bash
pip install mypy
```

**VS Code Integration:**
1. Install Python extension
2. Enable "python.analysis.typeCheckingMode": "basic" or "strict"
3. MyPy errors appear in real-time as red underlines!

### 🎯 Command Line Usage

```bash
# Check single file
mypy your_ico_code.py

# Check entire project  
mypy src/

# Strict mode (recommended for ICO)
mypy --strict your_ico_code.py

# Show error codes
mypy --show-error-codes your_ico_code.py
```





---



## 9. What's Next?

### 🎯 **ICO Basics**
- [ICO Runtime introduction](src/examples/ico_runtime_basics.ipynb): Progress monitoring, printing and runtime architecture

### 🔄 **Multiprocessing**
- [Multi-processing example](src/examples/ico_mp_basic.py): Basic example of distributed computational flows
- [Parallel Multi-processing Pool example](src/examples/ico_mp_basic_pool.py): Distributed compute flows with parallel worker pools

### 🧠 **Machine Learning**
- [Linear Regression](src/examples/ml/ico_linear_regression.ipynb): ICO-based ML pipeline development
- [CIFAR-10 Classification with validation](src/examples/ml/cv/cifar/ico_cifar_complete_flow.ipynb): Complete CV pipeline replacing PyTorch DataLoader
- [CIFAR-10 Classification with worker pools](src/examples/ml/cv/cifar/ico_cifar_complete_flow_mp.py): Complete CV pipeline with parallel data processing workers

**Happy coding with ICO! 🎉**
